# Taller 1: CAPM en dos mercados

**Julián Duarte -- Valoración de Activos (DEFI 2)**

Este notebook reconstruye la base del expediente, audita datos, estima CAPM con errores HAC, calcula métricas de riesgo/liquidez/robustez, produce las siete visualizaciones obligatorias y cierra con recomendación para Doña Victoria.

In [ ]:
import sys, subprocess, importlib, os
from pathlib import Path

base_dir = Path.cwd()
for p in [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]:
    if (p / '01_datos').exists():
        base_dir = p
        break
datos_dir = base_dir / '01_datos'
graficos_dir = base_dir / '04_graficos'
scripts_dir = base_dir / '05_scripts'

print('Python:', sys.version)
for pkg in ['numpy','pandas','matplotlib','seaborn','statsmodels','yfinance','nbformat']:
    mod = importlib.import_module(pkg)
    print(pkg, getattr(mod, '__version__', 'ok'))

print('Entorno y versiones registrados.')

In [ ]:
# Ejecuta el pipeline reproducible: descarga precios/volumen, alinea calendarios,
# construye retornos/excesos, estima metricas y regenera tablas/graficos.
script_target = str(scripts_dir / 'rebuild_entregables.py')
exec(open(script_target).read())

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv(datos_dir / 'datos_taller_1_julian_duarte.csv', parse_dates=['Date'])
master = pd.read_csv(datos_dir / 'tabla_maestra_resultados.csv')
seleccion = pd.read_csv(datos_dir / 'evidencia_seleccion_activos.csv')

print(df.shape, df['Date'].min(), df['Date'].max())
display(seleccion)
display(master[['Ticker','Benchmark','Beta','CI_Beta_Low','CI_Beta_High','R2','Observed_Return','CAPM_Expected_Return']])

In [ ]:
activos = ['SOL','KOPN','MVIS','LIDR','GERN','CTMX','OCGN','BORR','KLXE','TTI']
auditoria = []
for t in activos:
    r = df[f'{t}_Return']
    auditoria.append({
        'Activo': t,
        'Inicio': df['Date'].min().date(),
        'Fin': df['Date'].max().date(),
        'Obs': len(r),
        'Faltantes': int(r.isna().sum()),
        'Duplicados fecha': int(df['Date'].duplicated().sum()),
        'Retornos cero (%)': round((r == 0).mean()*100, 2),
        'Volumen mediano': round(df[f'{t}_Volume'].median(), 0),
        '5 peores retornos': ', '.join(f'{x:.2%}' for x in r.nsmallest(5)),
        '5 mejores retornos': ', '.join(f'{x:.2%}' for x in r.nlargest(5)),
    })
display(pd.DataFrame(auditoria))

In [ ]:
print('Tabla maestra 1: CAPM')
display(master[['Ticker','Family','Benchmark','Alpha_Annual','Beta','SE_Beta','CI_Beta_Low','CI_Beta_High','P_Val_Beta','P_Val_Alpha','R2','R2_Adj','Observed_Return','CAPM_Expected_Return']])

print('Tabla maestra 2: riesgo, desempeno y liquidez')
display(master[['Ticker','Vol_Annual','Sharpe','Treynor','Sortino','MDD','VaR_95_Daily','CVaR_95_Daily','Best_Day','Worst_Day','Skewness','Kurtosis','Pct_Zero_Returns','Med_Volume','Med_Monetary_Volume','Amihud_Illiquidity','Beta_Minus','Beta_Dimson_Sum','Beta_Winsor','T_Beta_1']])

## Visualizaciones obligatorias

Las siete figuras quedan guardadas en `04_graficos/`:

1. Precios normalizados por expediente.
2. Dispersión CAPM con recta para todos los activos.
3. Forest plot de betas con IC HAC.
4. Beta móvil de 90 observaciones para todos los activos.
5. Mapa riesgo-retorno.
6. Drawdown acumulado para todos los activos.
7. Matriz de percentiles por familia.

In [ ]:
from IPython.display import Image, display
for filename in [
    'grafico_1_precios_normalizados.png',
    'grafico_2_recta_capm_todos.png',
    'grafico_3_forest_plot_betas.png',
    'grafico_4_beta_movil_todos.png',
    'grafico_5_mapa_riesgo_retorno.png',
    'grafico_6_drawdown_acumulado_todos.png',
    'grafico_7_matriz_percentiles.png',
]:
    imgPath = graficos_dir / filename
    print(imgPath)
    if imgPath.exists():
        display(Image(filename=str(imgPath)))

## Dictamen final

Doña Victoria no puede exigir 25% anual sin caídas: esa restricción contradice la evidencia de drawdowns, CVaR e iliquidez. Para conservación, se deben evitar posiciones con baja liquidez y pérdidas extremas. Para crecimiento, SOL solo debe mantenerse como exposición acotada al mercado cripto. Para especulación acotada, las acciones de bajo precio deben limitarse a porcentajes pequeños del portafolio y con pérdida máxima presupuestada.

La beta se interpreta dentro de cada benchmark: SOL frente al NCI, tecnología frente a XLK, biotecnología frente a XBI y energía frente a XLE. Una beta baja no se acepta como defensiva si viene acompañada de retornos cero, bajo volumen monetario o fuerte sensibilidad a Dimson/winsorización.